# 15-1: Repaso — Linear Regression (Simple and Multiple)

**Course:** Models of Statistical Analysis (MAE) — Universidad de los Andes  
**Instructor:** Prof. Alejandra Tabares  
**Week:** 15 — Final Review

## 1. Simple Linear Regression (SLR)

### Model

$$
Y_i = \beta_0 + \beta_1 x_i + \varepsilon_i, \qquad i = 1, \ldots, n
$$

### LINE Assumptions

| Letter | Assumption | How to check |
|---|---|---|
| **L** | **Linearity**: $E[Y|x]$ is linear in $x$ | Residuals vs fitted — no curve |
| **I** | **Independence**: observations are independent | Study design / Durbin-Watson |
| **N** | **Normality**: $\varepsilon_i \sim \mathcal{N}(0, \sigma^2)$ | QQ-plot, Shapiro-Wilk test |
| **E** | **Equal variance** (homoscedasticity): $\text{Var}(\varepsilon_i) = \sigma^2$ | Scale-location plot, Breusch-Pagan |

### OLS Estimators

$$
S_{xx} = \sum_{i=1}^n (x_i - \bar{x})^2, \quad S_{xy} = \sum_{i=1}^n (x_i - \bar{x})(y_i - \bar{y})
$$

$$
\hat{\beta}_1 = \frac{S_{xy}}{S_{xx}}, \qquad \hat{\beta}_0 = \bar{y} - \hat{\beta}_1 \bar{x}
$$

$$
\hat{\sigma}^2 = \frac{SS_E}{n-2} = \frac{\sum_i (y_i - \hat{y}_i)^2}{n-2}
$$

### Properties of OLS (under LINE)

- **Gauss-Markov theorem**: OLS is BLUE (Best Linear Unbiased Estimator)
- $\hat{\beta}_j$ are unbiased: $E[\hat{\beta}_j] = \beta_j$
- $\text{Var}(\hat{\beta}_1) = \sigma^2 / S_{xx}$
- Under normality: $\hat{\beta}_j \sim \mathcal{N}(\beta_j, \text{Var}(\hat{\beta}_j))$

## 2. Multiple Linear Regression (MLR)

### Model

$$
\mathbf{y} = \mathbf{X}\boldsymbol{\beta} + \boldsymbol{\varepsilon}, \qquad \boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0}, \sigma^2 \mathbf{I}_n)
$$

where $\mathbf{X}$ is the $n \times (p+1)$ design matrix (including the intercept column).

### OLS Estimator

$$
\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}
$$

$$
\text{Var}(\hat{\boldsymbol{\beta}}) = \sigma^2 (\mathbf{X}^\top \mathbf{X})^{-1}
$$

### Interpretation

$\hat{\beta}_j$ is the estimated change in $E[Y]$ for a one-unit increase in $x_j$, **holding all other predictors fixed**.

### Inference

| Test | Hypotheses | Statistic | Distribution |
|---|---|---|---|
| **t-test** (individual coefficient) | $H_0: \beta_j = 0$ | $t_j = \hat{\beta}_j / \hat{\text{SE}}(\hat{\beta}_j)$ | $t_{n-p-1}$ |
| **F-test** (global) | $H_0: \beta_1 = \cdots = \beta_p = 0$ | $F = (SS_R/p)/(SS_E/(n-p-1))$ | $F_{p,\,n-p-1}$ |
| **Partial F-test** | $H_0$: subset of coefficients $= 0$ | $(SS_{E,R}-SS_{E,F})/(p_F-p_R) \;/\; \hat{\sigma}^2_F$ | $F_{p_F-p_R,\,n-p_F-1}$ |

### ANOVA Table for MLR

| Source | SS | df | MS | F |
|---|---|---|---|---|
| Regression | $SS_R = \hat{\boldsymbol{\beta}}^\top \mathbf{X}^\top \mathbf{y} - n\bar{y}^2$ | $p$ | $MS_R = SS_R/p$ | $F = MS_R/MS_E$ |
| Error | $SS_E = \mathbf{y}^\top\mathbf{y} - \hat{\boldsymbol{\beta}}^\top\mathbf{X}^\top\mathbf{y}$ | $n-p-1$ | $MS_E = SS_E/(n-p-1)$ | |
| Total | $SS_T = \mathbf{y}^\top\mathbf{y} - n\bar{y}^2$ | $n-1$ | | |

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(99)
n = 150

# Simulate dataset: exam score ~ hours_studied + prev_grade + sleep
hours   = np.random.uniform(1, 10, n)
grade   = np.random.normal(70, 10, n)          # previous grade (0-100)
sleep   = np.random.uniform(4, 9, n)           # hours of sleep

# True model: score = 20 + 4*hours + 0.3*grade + 1.5*sleep + noise
score = 20 + 4.0 * hours + 0.3 * grade + 1.5 * sleep + np.random.normal(0, 5, n)
score = np.clip(score, 0, 100)

df = pd.DataFrame({'score': score, 'hours': hours, 'grade': grade, 'sleep': sleep})
print('Simulated dataset — first 5 rows')
print(df.head().round(2).to_string(index=False))

# ── Simple Linear Regression: score ~ hours ───────────────────────────────────
print('\n' + '='*60)
print('SIMPLE LINEAR REGRESSION: score ~ hours')
print('='*60)
slr = smf.ols('score ~ hours', data=df).fit()
print(slr.summary())

# ── Multiple Linear Regression: score ~ hours + grade + sleep ─────────────────
print('\n' + '='*60)
print('MULTIPLE LINEAR REGRESSION: score ~ hours + grade + sleep')
print('='*60)
mlr = smf.ols('score ~ hours + grade + sleep', data=df).fit()
print(mlr.summary())

In [ ]:
# ── Confidence intervals and prediction intervals ─────────────────────────────

print('MLR — 95% Confidence Intervals for Coefficients')
print(mlr.conf_int(alpha=0.05).rename(columns={0: 'CI lower 2.5%', 1: 'CI upper 97.5%'}).round(4))

# Predict for a new observation
new_obs = pd.DataFrame({'hours': [6.0], 'grade': [72.0], 'sleep': [7.0]})
pred = mlr.get_prediction(new_obs)
pred_summary = pred.summary_frame(alpha=0.05)

print('\nPrediction for new observation: hours=6, grade=72, sleep=7')
print('─'*55)
print(f"  Predicted score (point estimate): {pred_summary['mean'].values[0]:.2f}")
print(f"  95% CI for E[Y|x]:  [{pred_summary['mean_ci_lower'].values[0]:.2f}, "
      f"{pred_summary['mean_ci_upper'].values[0]:.2f}]")
print(f"  95% PI for Y*:      [{pred_summary['obs_ci_lower'].values[0]:.2f}, "
      f"{pred_summary['obs_ci_upper'].values[0]:.2f}]")
print('\n  Note: The PI is always wider than the CI — it accounts for')
print('  both uncertainty in the mean and individual-level variance.')

## 3. Diagnostics Checklist

After fitting a linear regression, always run the following diagnostic checks:

| Plot | What to look for | Problem if... |
|---|---|---|
| **Residuals vs Fitted** | Random scatter around 0 | Systematic curve → nonlinearity; fan shape → heteroscedasticity |
| **Normal QQ-plot** | Points fall on the diagonal | Heavy tails, S-curve → non-normality |
| **Scale-Location** | Flat horizontal line, random scatter | Upward trend → heteroscedasticity |
| **Leverage plot** (Cook's D) | No points with high leverage AND large residual | Influential observations distort the fit |

**VIF (Variance Inflation Factor):**

$$
\text{VIF}_j = \frac{1}{1 - R_j^2}
$$

where $R_j^2$ is the $R^2$ from regressing $x_j$ on all other predictors.

- VIF = 1: no multicollinearity
- VIF 1–5: moderate (usually acceptable)
- VIF > 10: serious multicollinearity — SEs are inflated

In [ ]:
# ── 2×2 Diagnostic Plot Panel ─────────────────────────────────────────────────

import scipy.stats as stats

fitted   = mlr.fittedvalues
residuals = mlr.resid
std_resid = residuals / residuals.std()
leverage  = mlr.get_influence().hat_matrix_diag
cooks_d   = mlr.get_influence().cooks_distance[0]

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle('MLR Diagnostic Plots', fontsize=14, y=1.01)

# 1. Residuals vs Fitted
ax = axes[0, 0]
ax.scatter(fitted, residuals, alpha=0.5, s=20, color='steelblue')
ax.axhline(0, color='red', linewidth=1, linestyle='--')
# Smooth trend line
z = np.polyfit(fitted, residuals, 2)
p = np.poly1d(z)
x_sorted = np.sort(fitted)
ax.plot(x_sorted, p(x_sorted), color='orange', linewidth=1.5, label='lowess approx')
ax.set_xlabel('Fitted values', fontsize=10)
ax.set_ylabel('Residuals', fontsize=10)
ax.set_title('Residuals vs Fitted', fontsize=11)
ax.legend(fontsize=8)

# 2. Normal QQ-plot
ax = axes[0, 1]
(osm, osr), (slope, intercept, r) = stats.probplot(residuals, dist='norm')
ax.scatter(osm, osr, alpha=0.5, s=20, color='steelblue')
ax.plot(osm, slope * np.array(osm) + intercept, color='red', linewidth=1.5)
ax.set_xlabel('Theoretical quantiles', fontsize=10)
ax.set_ylabel('Sample quantiles', fontsize=10)
ax.set_title('Normal Q-Q Plot', fontsize=11)

# 3. Scale-Location (sqrt |standardized residuals| vs fitted)
ax = axes[1, 0]
sqrt_abs_resid = np.sqrt(np.abs(std_resid))
ax.scatter(fitted, sqrt_abs_resid, alpha=0.5, s=20, color='steelblue')
z2 = np.polyfit(fitted, sqrt_abs_resid, 1)
p2 = np.poly1d(z2)
ax.plot(x_sorted, p2(x_sorted), color='red', linewidth=1.5)
ax.set_xlabel('Fitted values', fontsize=10)
ax.set_ylabel(r'$\sqrt{|\mathrm{Std.\,residuals}|}$', fontsize=10)
ax.set_title('Scale-Location', fontsize=11)

# 4. Residuals vs Leverage (Cook's D as bubble size)
ax = axes[1, 1]
ax.scatter(leverage, std_resid, s=cooks_d * 3000 + 5,
           alpha=0.5, color='steelblue')
ax.axhline(0, color='red', linewidth=1, linestyle='--')
ax.axhline(2, color='grey', linewidth=0.8, linestyle=':')
ax.axhline(-2, color='grey', linewidth=0.8, linestyle=':')
# Annotate top influential points
top_idx = np.argsort(cooks_d)[-3:]
for idx in top_idx:
    ax.annotate(str(idx), (leverage[idx], std_resid[idx]),
                fontsize=8, color='darkred')
ax.set_xlabel('Leverage', fontsize=10)
ax.set_ylabel('Standardized residuals', fontsize=10)
ax.set_title("Residuals vs Leverage\n(bubble size ∝ Cook's D)", fontsize=11)

plt.tight_layout()
plt.show()

# VIF
X_vif = df[['hours', 'grade', 'sleep']].copy()
X_vif = sm.add_constant(X_vif)
vif_data = pd.DataFrame({
    'Variable': X_vif.columns[1:],
    'VIF': [variance_inflation_factor(X_vif.values, i+1)
            for i in range(X_vif.shape[1]-1)]
})
print('\nVariance Inflation Factors')
print(vif_data.round(3).to_string(index=False))

## 4. Variable Selection Methods

### Information Criteria

| Criterion | Formula | Favors |
|---|---|---|
| AIC | $-2\ell + 2p$ | Predictive accuracy |
| BIC | $-2\ell + p\ln n$ | Parsimony (penalizes more for large $n$) |
| Adjusted $R^2$ | $1-(1-R^2)(n-1)/(n-p-1)$ | Explains variance, adjusted for $p$ |

### Stepwise Selection

| Strategy | Process | Risk |
|---|---|---|
| Forward | Start empty, add predictors one by one | May miss important combinations |
| Backward | Start full, remove predictors one by one | Can fail if $p > n$ |
| Stepwise | Alternates forward and backward | Multiple testing inflation |

> **Caution:** Stepwise selection inflates Type I error and produces optimistic $R^2$. Use for exploration, not inference.

### Regularization (Shrinkage)

Both Ridge and Lasso solve a penalized least squares problem:

$$
\hat{\boldsymbol{\beta}}_{\text{Ridge}} = \arg\min_{\boldsymbol{\beta}} \left\{ \|\mathbf{y} - \mathbf{X}\boldsymbol{\beta}\|^2 + \lambda\|\boldsymbol{\beta}\|_2^2 \right\}
$$

$$
\hat{\boldsymbol{\beta}}_{\text{Lasso}} = \arg\min_{\boldsymbol{\beta}} \left\{ \|\mathbf{y} - \mathbf{X}\boldsymbol{\beta}\|^2 + \lambda\|\boldsymbol{\beta}\|_1 \right\}
$$

Ridge has a closed form: $\hat{\boldsymbol{\beta}}_{\text{Ridge}} = (\mathbf{X}^\top\mathbf{X} + \lambda\mathbf{I})^{-1}\mathbf{X}^\top\mathbf{y}$.

**Key difference:** Lasso sets some coefficients exactly to zero (variable selection); Ridge shrinks all toward zero but never exactly.

In [ ]:
# ── OLS vs Ridge vs Lasso comparison ─────────────────────────────────────────

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

np.random.seed(77)
n2 = 200
p2 = 10  # some true predictors, some noise

X_all = np.random.normal(0, 1, (n2, p2))
# Only first 4 predictors have non-zero true coefficients
beta_true = np.array([3.0, -2.0, 1.5, -1.0, 0, 0, 0, 0, 0, 0])
y2 = X_all @ beta_true + np.random.normal(0, 2, n2)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)

lam = 1.0

ols   = LinearRegression().fit(X_scaled, y2)
ridge = Ridge(alpha=lam).fit(X_scaled, y2)
lasso = Lasso(alpha=lam / n2, max_iter=10000).fit(X_scaled, y2)

coef_df = pd.DataFrame({
    'Variable':   [f'x{j+1}' for j in range(p2)],
    'True':       beta_true,
    'OLS':        ols.coef_,
    'Ridge':      ridge.coef_,
    'Lasso':      lasso.coef_
})
print('Coefficient comparison: OLS vs Ridge vs Lasso')
print('(All predictors standardized, lambda=1)')
print('='*60)
print(coef_df.round(3).to_string(index=False))

# Cross-validated R^2
r2_ols   = cross_val_score(LinearRegression(), X_scaled, y2, cv=5,
                            scoring='r2').mean()
r2_ridge = cross_val_score(Ridge(alpha=lam), X_scaled, y2, cv=5,
                            scoring='r2').mean()
r2_lasso = cross_val_score(Lasso(alpha=lam/n2, max_iter=10000), X_scaled, y2,
                            cv=5, scoring='r2').mean()
print(f'\n5-fold CV R²: OLS={r2_ols:.3f}, Ridge={r2_ridge:.3f}, Lasso={r2_lasso:.3f}')

# Plot coefficient paths
fig, ax = plt.subplots(figsize=(9, 4))
x_pos = np.arange(p2)
width = 0.25
ax.bar(x_pos - width, coef_df['True'],  width, label='True',  color='grey',      alpha=0.7)
ax.bar(x_pos,         coef_df['OLS'],   width, label='OLS',   color='steelblue', alpha=0.8)
ax.bar(x_pos + width, coef_df['Ridge'], width, label='Ridge', color='darkorange',alpha=0.8)
ax2 = ax.twinx()
ax2.scatter(x_pos, coef_df['Lasso'], color='crimson', s=70, zorder=5, label='Lasso')
ax2.axhline(0, color='crimson', linewidth=0.5, linestyle=':')
ax2.set_ylabel('Lasso coefficient', color='crimson', fontsize=10)
ax.set_xticks(x_pos)
ax.set_xticklabels(coef_df['Variable'])
ax.set_ylabel('Coefficient value', fontsize=10)
ax.set_title('OLS vs Ridge vs Lasso Coefficient Estimates', fontsize=12)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper right')
plt.tight_layout()
plt.show()